In [1]:
!pip install python-dotenv pandas

  Using cached pandas-2.3.3-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.3.4-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-2.3.3-cp311-cp311-win_amd64.whl (11.3 MB)
Using cached numpy-2.3.4-cp311-cp311-win_amd64.whl (13.1 MB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)

   ---------------------------------------- 0/5 [pytz]
   -------- ------------------------------- 1/5 [tzdata]
   ---------------- ----------------------- 2/5 [python-dotenv]
   ---------------- ----------------------- 2/5 [python-dotenv]
   ------------------------ --------------- 3/5 [numpy]
   ------------------------ --------------- 3/5 [numpy]
   ------------------------ --------------- 3/5 [numpy]
   ------------------------ --------------- 3/5 [numpy]
   -----------------------

In [3]:
import os
import sys
import urllib.request
import datetime
import time
import json
import pandas as pd
import datetime
import re
from dotenv import load_dotenv

In [4]:
load_dotenv()

# 환경변수에서 불러오기
client_id = os.getenv("NAVER_CLIENT_ID")
client_secret = os.getenv("NAVER_CLIENT_SECRET")


In [5]:
#검색 키워드
search_keyword = '서울 "사건 발생" 경찰 '

#성범죄: 온라인 
crimes = ['성폭행', '흉기 난동', '살인', '강도', '절도', '묻지마', '폭행']

In [10]:
encText = urllib.parse.quote(str(search_keyword+ crimes[0]))
today = datetime.datetime.now().strftime('%Y-%m-%d')

# 테스트용 첫 페이지만 가져오기
url = "https://openapi.naver.com/v1/search/news.json?query=" + encText + "&start=" + str(1) + "&display=100&sort=sim"

In [11]:
request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id",client_id)
request.add_header("X-Naver-Client-Secret",client_secret)

In [12]:
response = urllib.request.urlopen(request)
rescode = response.getcode()

if(rescode==200):
    response_body = response.read()
    print('request Success')
    # print(response_body.decode('utf-8'))
else:
    print("Error Code:" + rescode)


response_result = response_body.decode('utf-8')
response_result = json.loads(response_result)

request Success


In [13]:
response_result

{'lastBuildDate': 'Mon, 10 Nov 2025 17:31:54 +0900',
 'total': 4281,
 'start': 1,
 'display': 100,
 'items': [{'title': '‘1인2역’ 협박으로 여성 <b>성폭행</b>한 30대 검거…피해자 100여명 추정',
   'originallink': 'https://www.segye.com/newsView/20251031513814?OutUrl=naver',
   'link': 'https://n.news.naver.com/mnews/article/022/0004079287?sid=102',
   'description': '협박해 <b>성폭행</b>한 30대 남성이 <b>경찰</b>에 적발됐다. 31일 <b>경찰</b> 등에 따르면 <b>서울</b> 서초<b>경찰</b>서는 2022년쯤부터... 피해자 측 법률대리인은 피해자가 자신을 가해자로 착각해 박씨의 요구에 응할 수밖에 없었다며, <b>사건 발생</b> 3년이... ',
   'pubDate': 'Fri, 31 Oct 2025 18:02:00 +0900'},
  {'title': '[단독] &quot;욕설 들으며 강압적 <b>성폭행</b> 당했다&quot;더니…법정서 &quot;미안, 다 거짓...',
   'originallink': 'https://lawtalknews.co.kr/article/Y3SD7UMYPRP7',
   'link': 'https://lawtalknews.co.kr/article/Y3SD7UMYPRP7',
   'description': '어플리케이션으로 만난 남성에게 <b>성폭행</b>을 당했다며 상세하고 구체적인 피해 사실을 진술했던 20대 여성이, 법정에 이르러 자신의 모든 진술이 거짓이었다고 전면 번복하는 <b>사건</b>이 <b>발생</b>했다. 피고인의 강압이나 폭력은... ',
   'pubDate': 'Sat, 25 Oct 2025 15:30:00 +0900'},
  {'tit

In [14]:
print(response_result['items'][1]['description'])

어플리케이션으로 만난 남성에게 <b>성폭행</b>을 당했다며 상세하고 구체적인 피해 사실을 진술했던 20대 여성이, 법정에 이르러 자신의 모든 진술이 거짓이었다고 전면 번복하는 <b>사건</b>이 <b>발생</b>했다. 피고인의 강압이나 폭력은... 


In [15]:
import re
from email.utils import parsedate_to_datetime
import pandas as pd

data = response_result  # 네이버 응답 dict

# 1. 광역 단위
REGIONS = [
    '서울', '부산', '인천', '대구', '대전', '광주', '울산', '세종',
    '경기', '강원', '충북', '충남', '전북', '전남', '경북', '경남', '제주'
]

# 2. 세부 지역 패턴 (자주 나오는 구/동/시)
#   - ~구, ~동, ~읍, ~면 을 우선 뽑습니다.
DETAIL_PATTERN = re.compile(r'([가-힣]+구|[가-힣]+동|[가-힣]+읍|[가-힣]+면)')

def clean_html(text: str) -> str:
    if not text:
        return ''
    return re.sub(r'</?b>', '', text)

def extract_region(text: str):
    for r in REGIONS:
        if r in text:
            return r
    return None

def extract_detail(text: str):
    """
    제목/본문에서 처음 나오는 '미아동', '관악구' 같은 세부 행정 단위를 뽑습니다.
    여러 개가 나오면 첫 번째만.
    """
    m = DETAIL_PATTERN.search(text)
    if m:
        return m.group(1)
    return None

def parse_pubdate(pubdate_str: str):
    try:
        dt = parsedate_to_datetime(pubdate_str)
        return dt.date()
    except Exception:
        return None

rows = []

for item in data.get('items', []):
    title = clean_html(item.get('title', ''))
    desc = clean_html(item.get('description', ''))
    link = item.get('link', '')
    pubdate = item.get('pubDate', '')

    text_all = f"{title} {desc}"

    date = parse_pubdate(pubdate)
    region = extract_region(text_all)
    detail = extract_detail(text_all)

    rows.append({
        'date': date,
        'region': region,     # 서울/부산/경기...
        'detail': detail,     # 미아동/관악구/신림동...
        'title': title,
        'description': desc,
        'link': link
    })

df = pd.DataFrame(rows)
df


,date,region,detail,title,description,link
0,2025-10-31,서울,따르면,‘1인2역’ 협박으로 여성 성폭행한 30대 검거…피해자 100여명 추정,협박해 성폭행한 30대 남성이 경찰에 적발됐다. 31일 경찰 등에 따르면 서울 서초...,https://n.news.naver.com/mnews/article/022/000...
1,2025-10-25,None,전면,[단독] &quot;욕설 들으며 강압적 성폭행 당했다&quot;더니…법정서 &quo...,어플리케이션으로 만난 남성에게 성폭행을 당했다며 상세하고 구체적인 피해 사실을 진술...,https://lawtalknews.co.kr/article/Y3SD7UMYPRP7
2,2025-10-27,서울,None,2년 전 'BJ아영 의문사' 재조명..현직 변호사 &quot;가장 답답한건 주캄보디...,하지만 사건 발생 2년이 훌쩍 지난 지금까지도 진실은 밝혀지지 않았고 무엇보다 용의...,https://radio.ytn.co.kr/program/?f=2&id=105536...
3,2025-10-27,서울,None,2년 전 'BJ아영 의문사' 재조명..현직 변호사 &quot;가장 답답한건 주캄보디...,하지만 사건 발생 2년이 훌쩍 지난 지금까지도 진실은 밝혀지지 않았고 무엇보다 용의...,https://n.news.naver.com/mnews/article/052/000...
4,2025-10-24,서울,None,"기막힌 언론통제…“한국 여성 성추행한 베트남 차관, 현지 보도 0건”...",BBC 베트남은 지난 21일(현지시간) “지난달 11일 호앙 쑤언 찌엔 베트남 국방...,https://n.news.naver.com/mnews/article/081/000...
...,...,...,...,...,...,...
95,2025-03-31,서울,따르면,"'성폭행 혐의' 장제원 전 의원 고소인측, 동영상 등 증거 제출",따르면 성폭행 사건은 2015년 11월 18일 자정 무렵부터 오전 8시 30분 사이...,https://www.slist.kr/news/articleView.html?idx...
96,2025-03-31,서울,따르면,"'성폭행 혐의' 장제원 전 의원 고소인 측, 동영상 등 증거 제출",따르면 성폭행 사건은 2015년 11월 18일 자정 무렵부터 오전 8시 30분 사이...,https://n.news.naver.com/mnews/article/055/000...
97,2025-03-31,서울,따르면,"장제원 전 의원 성폭행 혐의 관련 고소인 측, 증거 자료 제출",A씨 측에 따르면 사건은 2015년 11월 18일 자정 무렵부터 오전 8시 30분 ...,https://www.topstarnews.net/news/articleView.h...
98,2025-03-31,서울,None,"'비서 성폭행 혐의' 장제원 고소인 측, 동영상 증거 제출... &quot;추행 시...",상대로 성폭행을 저질렀다는 의혹을 받는다.또한 장 전 의원은 사건 발생 이후 A씨에...,https://www.insight.co.kr/news/497310


In [ ]:
import datetime
from dateutil import parser

dt = parser.parse(response_result['items']['pubDate'])
date = dt.date()  # 2025-09-15

In [ ]:
for item in response_result['items']:
    title = item['title']
    desc = item['description']
    pub_date = parser.parse(item['pubDate']).date()
    region = extract_region(title + desc)

    print(pub_date, region, title)
